# IE686 Capstone: Context Engineering with Deep Agents and Skill Files
This notebook builds a proposal-writing agent for the IE686 student projects.

- Target course: **IE686 Large Language Models and Agents (FSS 2026)**
- Capstone focus: **skill files, context engineering, deep agents, subagent isolation, and LangSmith traces**
- End result: a deep agent that turns a suggested topic into a research-backed project proposal draft

Working references:
- [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/index)
- [Deep Agents skills](https://docs.langchain.com/oss/python/deepagents/skills)
- [Deep Agents subagents](https://docs.langchain.com/oss/python/deepagents/subagents)
- [Trace Deep Agents in LangSmith](https://docs.langchain.com/langsmith/trace-deep-agents)

Local course materials used for the rubric:
- `IE686_LA_06_IntroStudentProjects.pdf`
- `IE685_LA_06_ContextEngineering.pptx`


## What this lab emphasizes

1. **Skill files as natural-language control surfaces**: students inspect and edit on-disk `SKILL.md` files.
2. **Context engineering as architecture**: write context, select context, compress context, isolate context.
3. **Progressive build-up**: research memo -> outline -> critique -> final orchestrated proposal.
4. **Traceability**: use LangSmith to inspect delegation, tool calls, and revision loops.


## How to use this notebook

- Run top-to-bottom the first time.
- Keep the checked-in skill files open in another editor tab while you experiment.
- For each comparison block, keep the prompt fixed and change only the skill configuration.
- The default example topic is **creating a secretary agent using OpenClaw**, but the final cells let you swap in your own project topic.


## 1. Setup and preflight

This notebook assumes live web research and LangSmith tracing.

Required environment variables:
- `OPENAI_API_KEY`
- `TAVILY_API_KEY`
- `LANGSMITH_API_KEY`

Optional:
- `DEEPAGENT_MODEL` to override the default model (`openai:gpt-5-mini`)
- `LANGSMITH_PROJECT` to set a custom tracing project name


In [39]:
%pip install -qU deepagents==0.4.11 langchain==1.2.12 langchain-openai==1.1.11 langsmith==0.7.6 tavily-python==0.7.23 python-dotenv pydantic requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


In [40]:
import json
import os
import re
import shutil
import textwrap
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.tools import tool
from langchain_core.tracers.context import collect_runs
from langchain_openai import ChatOpenAI
from langsmith import Client
from pydantic import BaseModel, Field
from tavily import TavilyClient

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

load_dotenv()

ROOT = Path.cwd()
LAB_ROOT = ROOT / "deepagents_lab"
SKILLS_ROOT = LAB_ROOT / "skills"
WORKSPACE_ROOT = LAB_ROOT / "workspace"

SHARED_SKILLS = "/skills/shared"
MAIN_SKILLS = "/skills/main"
RESEARCH_SKILLS = "/skills/research"
REVIEW_SKILLS = "/skills/review"

MODEL_NAME = os.getenv("DEEPAGENT_MODEL", "openai:gpt-5-mini")
BASELINE_MODEL_NAME = MODEL_NAME.split(":", 1)[1] if MODEL_NAME.startswith("openai:") else MODEL_NAME

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "IE686-DeepAgents-Proposal-Lab")

print("Project root:", ROOT)
print("Lab root:", LAB_ROOT)
print("Skills root:", SKILLS_ROOT)
print("Workspace root:", WORKSPACE_ROOT)
print("Deep agent model:", MODEL_NAME)
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))


Project root: /Users/aaronsteiner/Documents/GitHub/llmsandagents
Lab root: /Users/aaronsteiner/Documents/GitHub/llmsandagents/deepagents_lab
Skills root: /Users/aaronsteiner/Documents/GitHub/llmsandagents/deepagents_lab/skills
Workspace root: /Users/aaronsteiner/Documents/GitHub/llmsandagents/deepagents_lab/workspace
Deep agent model: openai:gpt-5-mini
LangSmith project: LLM and agents


In [41]:
required_env = ["OPENAI_API_KEY", "TAVILY_API_KEY", "LANGSMITH_API_KEY"]
env_rows = [
    {
        "variable": name,
        "present": bool(os.getenv(name)),
        "preview": (os.getenv(name)[:6] + "...") if os.getenv(name) else "",
    }
    for name in required_env
]

skill_rows = []
for skill_file in sorted(SKILLS_ROOT.rglob("SKILL.md")):
    skill_rows.append(
        {
            "skill": skill_file.parent.name,
            "path": str(skill_file.relative_to(ROOT)),
            "bytes": skill_file.stat().st_size,
        }
    )

print("Environment preflight")
display(pd.DataFrame(env_rows))
print("\nChecked-in skill files")
display(pd.DataFrame(skill_rows))

if not all(row["present"] for row in env_rows):
    raise ValueError("Missing one or more required API keys. Add them to your environment or .env file before continuing.")


Environment preflight


,variable,present,preview
0,OPENAI_API_KEY,True,sk-pro...
1,TAVILY_API_KEY,True,tvly-q...
2,LANGSMITH_API_KEY,True,lsv2_p...



Checked-in skill files


,skill,path,bytes
0,proposal-writer,deepagents_lab/skills/main/proposal-writer/SKI...,1998
1,benchmark-research,deepagents_lab/skills/research/benchmark-resea...,1872
2,proposal-critic,deepagents_lab/skills/review/proposal-critic/S...,1113
3,project-outline-rubric,deepagents_lab/skills/shared/project-outline-r...,1115


We intentionally use a **filesystem-backed deep agent workspace** rooted at `deepagents_lab/`.

Why:
- skills live on disk and can be edited directly,
- agents can write research memos and drafts to `/workspace/...`,
- `virtual_mode=True` gives stable POSIX-style paths (`/skills/...`, `/workspace/...`) across macOS and Windows.


## 2. Inspect the checked-in skill library

Before running any agent, look at the skills that will steer it.
The point of this exercise is that **natural-language instructions on disk** can change agent behavior without changing Python code.


In [42]:
def list_skills() -> pd.DataFrame:
    rows = []
    for skill_file in sorted(SKILLS_ROOT.rglob("SKILL.md")):
        text = skill_file.read_text(encoding="utf-8")
        name_match = re.search(r"^name:\s*(.+)$", text, flags=re.MULTILINE)
        desc_match = re.search(r"^description:\s*(.+)$", text, flags=re.MULTILINE)
        rows.append(
            {
                "lane": skill_file.parent.parent.name,
                "skill": name_match.group(1).strip() if name_match else skill_file.parent.name,
                "description": desc_match.group(1).strip() if desc_match else "",
                "path": str(skill_file.relative_to(ROOT)),
            }
        )
    return pd.DataFrame(rows)


list_skills()


,lane,skill,description,path
0,main,proposal-writer,Use when turning research notes into a structu...,deepagents_lab/skills/main/proposal-writer/SKI...
1,research,benchmark-research,Use when the task requires web research about ...,deepagents_lab/skills/research/benchmark-resea...
2,review,proposal-critic,Use when critiquing or revising a student proj...,deepagents_lab/skills/review/proposal-critic/S...
3,shared,project-outline-rubric,"Use when drafting, reviewing, or revising a st...",deepagents_lab/skills/shared/project-outline-r...


In [43]:
def show_skill(skill_name: str) -> None:
    matches = list(SKILLS_ROOT.rglob(f"{skill_name}/SKILL.md"))
    if not matches:
        raise FileNotFoundError(f"No skill named {skill_name!r} found.")

    skill_file = matches[0]
    print(f"=== {skill_file.relative_to(ROOT)} ===")
    print(skill_file.read_text(encoding="utf-8"))


show_skill("benchmark-research")


=== deepagents_lab/skills/research/benchmark-research/SKILL.md ===
---
name: benchmark-research
description: Use when the task requires web research about prior systems, recent scientific writing, datasets, benchmarks, APIs, or evaluation metrics for a project topic. Focus on exact names, URLs, recency, suitability, and comparison value.
---

# Benchmark Research

Use this skill when the agent must gather external evidence before drafting a proposal.

## Workflow

1. Start with 2-4 focused web searches that combine the topic with terms such as:
   - benchmark
   - dataset
   - evaluation
   - baseline
   - prior work
   - paper
   - recent
2. Prioritize sources that help answer:
   - Which comparable systems already exist?
   - Which datasets, APIs, or environments are realistic?
   - Which benchmarks or metrics would make evaluation convincing?
3. Explicitly look for recent scientific writing that can make the proposal more concrete and current.
4. Prefer recent papers, benchmark writ

Try it out:
- Open `deepagents_lab/skills/research/benchmark-research/SKILL.md`.
- Which parts are general enough to reuse on other topics?
- Which parts are specific enough to change what the agent does?


## 3. Baseline: no skills, no explicit context engineering

Start with the simplest possible baseline: ask a strong model to write a proposal directly.
This is useful because it gives us a concrete “before” picture.


In [44]:
DEFAULT_TOPIC = (
    "Creating a secretary agent using OpenClaw: build an agent assistant that can help with inbox triage, "
    "calendar coordination, meeting preparation, and follow-up task tracking, then evaluate it against "
    "realistic office workflows and recent related systems."
)


def build_outline_request(topic: str) -> str:
    return textwrap.dedent(
        f"""
        Draft a project outline for the following student project topic:
        {topic}

        The outline must answer these questions:
        1. What is the problem you are solving?
        2. What data will you use, where will you get it, and how will you gather it?
        3. How will you solve the problem, which LLMs or methods will you use, and what is your multi-agent workflow?
        4. How will you measure success?

        Keep the answer concise and proposal-oriented.
        """
    ).strip()


baseline_model = ChatOpenAI(model=BASELINE_MODEL_NAME, temperature=0)
baseline_prompt = build_outline_request(DEFAULT_TOPIC)
baseline_response = baseline_model.invoke(baseline_prompt)
display(Markdown(baseline_response.content))


Project outline: Secretary agent using OpenClaw

1) Problem statement
- Build an automated “secretary” agent that reduces time and cognitive load for office workers by: triaging inboxes, coordinating calendars, preparing meeting materials, and tracking follow-up tasks.
- Key gaps addressed: fragmented tool interactions, low-quality single-response assistants, poor persistence of action items across email/calendar/meeting artifacts. The goal is a multi-agent pipeline that reliably identifies actions, schedules meetings, creates useful pre-reads/agendas, and ensures follow-up completion in realistic office workflows.

2) Data — what, source, and collection
- Email corpus (for triage & action extraction)
  - Public: Enron email dataset and Avocado dataset for initial training/benchmarking.
  - Synthetic: privacy-preserving synthetic emails generated to reflect modern corporate formats (calendar invites, threads, CCs, signatures).
  - Optional: with IRB/consent, anonymized exports from volunteer users (Gmail MBOX with PII redaction).
- Calendar data (scheduling, availability, conflicts)
  - Synthetic calendars created to simulate multiple roles/timezones/recurring events.
  - Public iCal samples and calendar export samples where available.
- Meeting artifacts (prep and follow-up)
  - Public meeting transcripts/corpora (AMI Meeting Corpus, ICSI) for agenda/summary models.
  - Synthetic meeting notes + audio->transcript pipeline if testing ASR integration (use Whisper or similar).
- Task and workflow labels
  - Manually labeled subsets: email intent (meeting, info, ack, task), priority, action items, proposed times, task completion status.
  - Small-scale human annotations for gold-standard evaluation (10–50 threads per scenario).
- Gathering method
  - Download public datasets, generate synthetic datasets with templates and LLM augmentation, and collect small consented samples for live user testing. Ensure PII redaction and secure storage.

3) Solution approach, models, and multi-agent workflow
- Architecture and platform
  - Orchestrate agents using OpenClaw to run specialized LLM agents, tool agents (calendar API, email API, vector DB), and a coordinator agent.
  - Persistence: vector DB (Milvus/Pinecone) for document embeddings & context; a small SQL/NoSQL store for task metadata and state.
  - RAG pipeline for long-context grounding (email history, calendar context, previous meeting notes).
- Models and tooling
  - LLMs: GPT-4o/GPT-4/Claude-2 or open models (Llama 3/Mistral) depending on access/cost. Use a stronger model for coordinator and summarization, smaller ones for classification where possible to save cost.
  - Embeddings: OpenAI/Cohere embeddings or open embedding models.
  - ASR (optional): Whisper for transcript generation.
  - Fine-tuning / instruction-tuning: few-shot prompting for extraction; small supervised fine-tune for intent/action classifiers if dataset permits.
  - Safety & privacy: PII detection/redaction model before storing or using content.
- Multi-agent workflow (high-level sequence)
  1. Ingest agent: receives new email/calendar events or meeting transcripts; stores raw and embeds context.
  2. InboxTriageAgent: classifies email type (meeting request, task, FYI, spam), priority, and extracts structured fields (action items, deadlines, proposed times).
  3. CoordinatorAgent: decides next steps based on triage outcome and user preferences (e.g., schedule meeting, auto-respond, convert to task).
  4. CalendarAgent: checks availability, negotiates times (proposes options), uses calendar API simulator (or real OAuth integration in later stages), and sends calendar invites or suggestions.
  5. PrepAgent: for scheduled meetings, compiles pre-read: agenda, relevant thread summary, action-item list, short talking points, and suggested questions.
  6. FollowUpAgent: after meetings/email actions, monitors for completion signals, pings assignees, converts action items into task entries with due dates, and escalates stale items.
  7. Interaction agent: formats messages to user (email drafts, Slack, notifications) and takes user feedback (accept/modify).
- Implementation notes
  - Use function-calling or tool-call interfaces for structured outputs (JSON action items, calendar calls).
  - Maintain conversational provenance: link each action back to source email and agent decision.
  - Allow human-in-the-loop: drafts require user approval for sending or scheduling by default; options for configurable autopilot rules.

4) Evaluation and success metrics
- Offline, synthetic/benchmark evaluation
  - Classification/Extraction: precision/recall/F1 on intent, action-item extraction, deadline detection using annotated test set.
  - Summarization quality: ROUGE, BERTScore for agenda/pre-read vs. gold summaries; supplement with edit-distance measures for action items.
  - Scheduling accuracy: conflict-free scheduling rate and acceptance simulation (percentage of proposed times accepted in simulated invitees), number of back-and-forth proposal rounds.
  - End-to-end simulation: percentage of workflows completed without human intervention when allowed; average number of manual touches required.
- Human user study (realistic office workflow)
  - Participants: 10–20 knowledge workers perform standardized scenarios (email triage, schedule a cross-team meeting, prepare for a meeting, close out tasks).
  - Measures:
    - Time saved (task completion time vs. manual baseline).
    - Task success rate (meeting scheduled, agenda accepted, tasks completed).
    - Usability and satisfaction: Likert ratings for helpfulness, trust, and clarity.
    - Preference/A-B test: compare multi-agent OpenClaw assistant vs. single-agent baseline (single LLM handling all tasks) and vs. rule-based baseline.
- Comparison to recent systems
  - Functional benchmarks vs. known features (Google Workspace/Microsoft Copilot style baselines): measure similar task outcomes (draft quality, scheduling latency, accuracy of action extraction).
  - Use publicly describable capabilities rather than proprietary internals; report comparative metrics (time saved, acceptance rates, user ratings).
- Success criteria (target thresholds)
  - Action extraction F1 > 0.80 on test set.
  - Scheduling conflict-free acceptance in ≤2 proposal rounds for ≥75% of simulated meetings.
  - Average time-to-complete tasks reduced by ≥30% vs. baseline.
  - User satisfaction score ≥4/5 in pilot study for usefulness and trust.
  - Demonstrated improvement over single-agent baseline across at least two of the above metrics.

Privacy, ethics, and deliverables
- Ensure PII redaction, secure storage, and consented user data only. Provide an opt-out and audit log for automated actions.
- Deliverables: codebase (OpenClaw orchestration and agents), synthetic + annotated dataset, evaluation scripts and reports, demo scenarios and a short usability study write-up.

Estimated timeline (8–12 weeks)
- Weeks 1–2: dataset assembly, annotation plan, initial prompts.
- Weeks 3–6: implement agents, RAG, integrations, basic UI for approvals.
- Weeks 7–9: offline evaluation, iterate models, human-in-the-loop tuning.
- Weeks 10–12: user study, comparative evaluation, final report and demo.

This plan is scoped to be implementable as a student project while producing measurable, reproducible results and a working multi-agent secretary prototype.

Typical baseline weaknesses:
- named benchmarks may be missing or unsupported,
- the data collection plan stays vague,
- the workflow is generic,
- there is no explicit trace of how the agent reached its choices.


## 4. Filesystem-backed file handling with deep agents

Before introducing skills, it helps to show that a deep agent can use a shared filesystem-backed workspace.

In this section the agent will:
- read a seed file from `/workspace/scratch/`,
- write a derived note back into `/workspace/scratch/`,
- return a short explanation of what it did.

This is the simplest way to show that file handling is already part of the agent runtime, even before we add skill files.


Why this matters:
- a proposal workflow usually needs intermediate artifacts such as memos, outlines, and review notes,
- those artifacts are easier to inspect on disk than when hidden inside a long chat history,
- later agent steps can discover and reuse files instead of requiring the full context to be pasted again.

In practice, this means that context engineering is not only about prompts. It is also about **where state lives** and **how the agent can find it again**.


The point of the demo is architectural:
- the agent does not need the file contents stuffed into a giant prompt,
- intermediate artifacts can live on disk,
- later agent steps can reuse those files as context.


In [45]:
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def tavily_search(query: str, max_results: int = 5, include_domains_csv: str = "") -> str:
    """Search the live web for current systems, benchmarks, datasets, APIs, and evaluation setups."""

    include_domains = [d.strip() for d in include_domains_csv.split(",") if d.strip()]
    raw = tavily_client.search(
        query=query,
        search_depth="advanced",
        topic="general",
        max_results=max_results,
        include_domains=include_domains or None,
        include_answer=False,
        include_raw_content="text",
    )

    rows = []
    for item in raw.get("results", [])[:max_results]:
        rows.append(
            {
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "content_snippet": (item.get("content", "") or "")[:500],
            }
        )
    return json.dumps(rows, indent=2)


@tool
def fetch_url_excerpt(url: str, max_chars: int = 3000) -> str:
    """Fetch the visible text of a webpage so the agent can inspect details after search."""

    response = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0 (compatible; IE686-DeepAgents-Lab/1.0)"},
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = " ".join(soup.stripped_strings)
    return text[:max_chars]


proposal_tools = [tavily_search, fetch_url_excerpt]


def make_backend() -> FilesystemBackend:
    return FilesystemBackend(root_dir=LAB_ROOT, virtual_mode=True)


def reset_workspace() -> None:
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    for child in WORKSPACE_ROOT.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

    variant_root = LAB_ROOT / "skill_variants"
    if variant_root.exists():
        shutil.rmtree(variant_root)

    for relative in ["research", "outline", "review", "final", "scratch"]:
        (WORKSPACE_ROOT / relative).mkdir(parents=True, exist_ok=True)


def extract_text_response(result: dict[str, Any]) -> str:
    last_message = result["messages"][-1]
    content = last_message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text") or block.get("content") or str(block))
            else:
                parts.append(str(block))
        return "\n".join(part for part in parts if part)
    return str(content)


def invoke_with_trace(agent, prompt: str, run_name: str) -> tuple[dict[str, Any], str | None]:
    with collect_runs() as runs_cb:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": prompt}]},
            config={"run_name": run_name},
        )

    run_url = None
    if runs_cb.traced_runs and os.getenv("LANGSMITH_API_KEY"):
        client = Client()
        run_url = client.get_run_url(run=runs_cb.traced_runs[0])
    return result, run_url


def build_lab_agent(skill_sources: list[str] | None):
    return create_deep_agent(
        model=MODEL_NAME,
        tools=proposal_tools,
        system_prompt=(
            "You help draft IE686 student project proposals. "
            "Use current evidence only."
        ),
        backend=make_backend(),
        skills=skill_sources,
        name="ie686_proposal_lab",
    )


### Built-in filesystem tools from `FilesystemMiddleware`

When you pass a `FilesystemBackend` into `create_deep_agent(...)`, Deep Agents already injects a file-oriented toolset through `FilesystemMiddleware`.
For this lab, that means we do **not** need to recreate custom file tools just to show discovery and editing.

The built-in tools that matter most here are:
- `ls`: discover files and directories
- `read_file`: inspect file contents
- `write_file`: create a new file
- `edit_file`: update an existing file

The middleware also provides `glob` and `grep` for broader discovery.

In other words, the basic create/read/update workflow is already part of the deep-agent runtime. The notebook helpers below are only used to seed and display files for teaching; they are **not** the agent interface.


In [46]:
def render_workspace_tree(relative_dir: str = "") -> str:
    clean = relative_dir.strip().strip("/")
    base = WORKSPACE_ROOT / Path(clean) if clean else WORKSPACE_ROOT
    if not base.exists():
        return f"/workspace/{clean} does not exist" if clean else "/workspace does not exist"

    lines = [f"$ ls -R /workspace/{clean}" if clean else "$ ls -R /workspace"]
    for path in sorted(base.rglob("*")):
        rel = path.relative_to(WORKSPACE_ROOT)
        kind = "dir" if path.is_dir() else "file"
        lines.append(f"{kind}: /workspace/{rel.as_posix()}")
    return "\n".join(lines)


def write_workspace_file(relative_path: str, content: str) -> Path:
    path = WORKSPACE_ROOT / Path(relative_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    return path

In [47]:
reset_workspace()

seed_file = write_workspace_file(
    "scratch/topic_brief.md",
    textwrap.dedent(
        """
        Topic: Creating a secretary agent using OpenClaw

        Goal:
        Build an agent assistant for inbox triage, calendar coordination, meeting preparation,
        and follow-up task tracking.
        """
    ).strip()
    + "\n",
)

print("=== Notebook-side setup ===\n")
print(render_workspace_tree("scratch"))
print("\n=== Seed file ===\n")
print(seed_file.read_text(encoding="utf-8"))


=== Notebook-side setup ===

$ ls -R /workspace/scratch
file: /workspace/scratch/topic_brief.md

=== Seed file ===

Topic: Creating a secretary agent using OpenClaw

Goal:
Build an agent assistant for inbox triage, calendar coordination, meeting preparation,
and follow-up task tracking.



In [48]:
def build_file_demo_agent():
    return create_deep_agent(
        model=MODEL_NAME,
        tools=[],
        backend=make_backend(),
        skills=None,
        system_prompt=(
            "You are demonstrating filesystem-backed context for an IE686 lab. "
            "Use the built-in filesystem tools from the backend. "
            "Start with `ls` before reading or modifying files, "
            "keep the edits concise, and explain which files you used."
        ),
        name="filesystem_demo_agent",
    )


reset_workspace()
write_workspace_file(
    "scratch/topic_brief.md",
    textwrap.dedent(
        """
        Topic: Creating a secretary agent using OpenClaw

        Goal:
        Build an agent assistant for inbox triage, calendar coordination, meeting preparation,
        and follow-up task tracking.

        Constraints:
        - The proposal should stay realistic for a student team.
        - The idea should reference recent related systems and evaluation setups.
        - Open questions should be separated from supported claims.
        """
    ).strip()
    + "\n",
)

file_demo_agent = build_file_demo_agent()
file_demo_prompt = textwrap.dedent(
    """
    First use `ls` to discover the workspace files.

    Then use `read_file` on /workspace/scratch/topic_brief.md.

    Then use `write_file` to create /workspace/scratch/proposal_scaffold.md with:
    1. a short prose summary of the project idea,
    2. three concrete follow-up questions the agent should research next.

    Finally, use `edit_file` once to add one more line at the end:
    "Open question: which office workflow should be the first evaluation scenario?"

    Return a short note describing which filesystem tools you used and in what order.
    """
).strip()

file_demo_result, file_demo_run_url = invoke_with_trace(
    file_demo_agent,
    file_demo_prompt,
    run_name="deepagents-filesystem-demo",
)

print(extract_text_response(file_demo_result))
print("\nFilesystem demo run:", file_demo_run_url or "LangSmith URL unavailable")


{'id': 'rs_09be665c84b16e290069ba754d02848195b7bd2561e1ba15aa', 'summary': [], 'type': 'reasoning'}
Actions performed (in order) and files used:
1. ls /workspace/scratch — discovered: /workspace/scratch/topic_brief.md
2. read_file /workspace/scratch/topic_brief.md — read the topic brief to base the proposal on
3. write_file /workspace/scratch/proposal_scaffold.md — created the proposal scaffold (summary + three follow-up questions)
4. read_file /workspace/scratch/proposal_scaffold.md — read the new file (required before editing)
5. edit_file /workspace/scratch/proposal_scaffold.md — appended: "Open question: which office workflow should be the first evaluation scenario?"

Files touched:
- /workspace/scratch/topic_brief.md (read)
- /workspace/scratch/proposal_scaffold.md (created and edited)

If you want changes to the scaffold wording or additional follow-up questions, say which direction and I'll update the file.

Filesystem demo run: https://smith.langchain.com/o/6ce8d416-0467-420e-b

In [49]:
seed_file = WORKSPACE_ROOT / "scratch" / "topic_brief.md"
scaffold_path = WORKSPACE_ROOT / "scratch" / "proposal_scaffold.md"

print("=== Seed file ===\n")
print(seed_file.read_text(encoding="utf-8"))
print("\n" + "=" * 100 + "\n")
print("=== Agent-written scaffold ===\n")
print(scaffold_path.read_text(encoding="utf-8"))


=== Seed file ===

Topic: Creating a secretary agent using OpenClaw

Goal:
Build an agent assistant for inbox triage, calendar coordination, meeting preparation,
and follow-up task tracking.

Constraints:
- The proposal should stay realistic for a student team.
- The idea should reference recent related systems and evaluation setups.
- Open questions should be separated from supported claims.



=== Agent-written scaffold ===

Project summary:
A student-team project to build a "secretary" agent using OpenClaw that performs inbox triage, calendar coordination, meeting preparation, and follow-up task tracking. The design emphasizes modular components (retrieval, intent classification, action generation) with an LLM backend and realistic privacy-aware connectors to email and calendar systems.

Follow-up questions to research next:
1. What email and calendar APIs (Gmail, Outlook, CalDAV) and permission models are practical for a student prototype, and what privacy constraints must we accou

Try it out:
- Edit `deepagents_lab/workspace/scratch/topic_brief.md`.
- Rerun only the filesystem demo cells.
- Check the trace to see whether the agent starts with `ls`, then moves through `read_file`, `write_file`, and `edit_file`.


## 5. Skills as natural-language steering

Now we give a deep agent access to live web tools and the checked-in skill folders.
The user prompt stays nearly the same; the main change is **what natural-language instructions the agent can discover on disk**.


### Exact skill files used in section 5

The skill-enabled comparison in this section uses these checked-in files:

- [`project-outline-rubric`](deepagents_lab/skills/shared/project-outline-rubric/SKILL.md)
- [`proposal-writer`](deepagents_lab/skills/main/proposal-writer/SKILL.md)
- [`benchmark-research`](deepagents_lab/skills/research/benchmark-research/SKILL.md)

The critique skill is part of the same lab library and is introduced later:

- [`proposal-critic`](deepagents_lab/skills/review/proposal-critic/SKILL.md)

These skills are also what introduce the more polished writing behavior:
executive-summary prose, recent scientific grounding, and source-backed references.

Read those files before rerunning the agent. This section is about showing that these natural-language files steer behavior without changing Python code.


In [50]:
reset_workspace()
no_skill_agent = build_lab_agent(skill_sources=None)
skilled_agent = build_lab_agent(skill_sources=[SHARED_SKILLS, MAIN_SKILLS, RESEARCH_SKILLS])

comparison_prompt = build_outline_request(DEFAULT_TOPIC)

no_skill_result, no_skill_run_url = invoke_with_trace(
    no_skill_agent,
    comparison_prompt,
    run_name="deepagents-no-skills",
)
skilled_result, skilled_run_url = invoke_with_trace(
    skilled_agent,
    comparison_prompt,
    run_name="deepagents-with-skills",
)

no_skill_text = extract_text_response(no_skill_result)
skilled_text = extract_text_response(skilled_result)

print("=== Deep agent without skills ===\n")
print(no_skill_text[:4000])
print("\n" + "=" * 100 + "\n")
print("=== Deep agent with checked-in skills ===\n")
print(skilled_text[:4000])


=== Deep agent without skills ===

{'id': 'rs_057ff307ded921a20069ba7554aeec819696e1aa30d8a34dcc', 'summary': [], 'type': 'reasoning'}
1) Problem statement
- Build a practical “secretary” agent that reduces human effort across four office tasks: inbox triage (prioritise, label, suggest actions), calendar coordination (find/offer times, negotiate constraints), meeting preparation (agendas, briefings, slide/checklist generation), and follow-up task tracking (extract action items, assign & remind).
- Key challenge: reliable end-to-end behaviour in realistic workflows (noisy emails, ambiguous requests, calendar constraints, privacy requirements) and measurable productivity gains vs existing tooling.

2) Data: what, sources, and collection plan
- Public corpora for development and offline evaluation:
  - Enron email dataset (de-identified) for inbox triage training/evaluation.
  - AMI Meeting Corpus or ICSI meeting transcripts for meeting-prep and summarization prototypes.
- Synthetic and c

In [51]:
print("No-skill LangSmith run:", no_skill_run_url or "LangSmith URL unavailable")
print("Skilled LangSmith run:", skilled_run_url or "LangSmith URL unavailable")


No-skill LangSmith run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d005a-500e-7bc3-930f-3ecf60faeca1?poll=true
Skilled LangSmith run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d005a-f103-70e0-9ab8-12b53381d3ca?poll=true


Try it out:
- Keep the prompt identical and edit one sentence in `deepagents_lab/skills/research/benchmark-research/SKILL.md`.
- Rerun only the skilled agent cell.
- Check whether the output becomes more concrete, more current, or more generic, and then inspect the trace in LangSmith.


## 6. Better skill design: weak trigger wording vs strong trigger wording

Skill quality matters twice:
1. the **description** controls whether the skill is loaded,
2. the **body** controls what the agent actually does after it loads the skill.

In this section we create a temporary weak variant of the research skill to show how vague trigger text hurts performance.


In [52]:
def make_weak_research_skill_variant() -> str:
    variant_root = LAB_ROOT / "skill_variants" / "weak_research"
    skill_dir = variant_root / "benchmark-research"
    if variant_root.exists():
        shutil.rmtree(variant_root)
    skill_dir.mkdir(parents=True, exist_ok=True)
    (skill_dir / "references").mkdir(parents=True, exist_ok=True)

    weak_skill = textwrap.dedent(
        """
        ---
        name: benchmark-research
        description: Use when the user needs help with something related to projects or information.
        ---

        # Weak benchmark research skill

        Try to be helpful.
        """
    ).strip() + "\n"

    source_reference = (
        SKILLS_ROOT
        / "research"
        / "benchmark-research"
        / "references"
        / "research_checklist.md"
    )
    shutil.copy2(source_reference, skill_dir / "references" / "research_checklist.md")
    (skill_dir / "SKILL.md").write_text(weak_skill, encoding="utf-8")
    return "/skill_variants/weak_research"


weak_research_source = make_weak_research_skill_variant()

strong_skill_file = SKILLS_ROOT / "research" / "benchmark-research" / "SKILL.md"
weak_skill_file = LAB_ROOT / "skill_variants" / "weak_research" / "benchmark-research" / "SKILL.md"

print("=== Strong description ===")
print(re.search(r"^description:\s*(.+)$", strong_skill_file.read_text(encoding="utf-8"), flags=re.MULTILINE).group(1))
print("\n=== Weak description ===")
print(re.search(r"^description:\s*(.+)$", weak_skill_file.read_text(encoding="utf-8"), flags=re.MULTILINE).group(1))


=== Strong description ===
Use when the task requires web research about prior systems, recent scientific writing, datasets, benchmarks, APIs, or evaluation metrics for a project topic. Focus on exact names, URLs, recency, suitability, and comparison value.

=== Weak description ===
Use when the user needs help with something related to projects or information.


In [53]:
weak_skill_agent = build_lab_agent(skill_sources=[SHARED_SKILLS, MAIN_SKILLS, weak_research_source])

weak_result, weak_run_url = invoke_with_trace(
    weak_skill_agent,
    comparison_prompt,
    run_name="deepagents-weak-skill",
)
weak_text = extract_text_response(weak_result)


In [54]:
print("Weak-skill LangSmith run:", weak_run_url or "LangSmith URL unavailable")
print("\nWeak-skill output preview:\n")
print(weak_text[:3000])


Weak-skill LangSmith run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d005b-920c-7c10-96c1-2331ab8f28ac?poll=true

Weak-skill output preview:

{'id': 'rs_0462a68cbcde7cdf0069ba75a6e6a881938b5a7a16cddeebcb', 'summary': [], 'type': 'reasoning'}
1) Problem statement — what you are solving
- Build a practical "secretary" assistant that automates common office workflows: inbox triage (priority routing, short replies, TODO extraction), calendar coordination (propose times, book/reschedule), meeting preparation (context aggregation, agenda and briefing), and follow-up task tracking (action extraction, reminders, status updates).
- Goal: reduce user cognitive load and time spent on routine coordination while maintaining high accuracy, privacy controls, and predictable, auditable actions.

2) Data: sources and collection plan
- Training / development corpora:
  - Enron email dataset (public) for realistic email language

In practice, the exact difference will vary by model and topic.
The main point is architectural: **skill descriptions are part of your retrieval layer for procedural knowledge**.


## 7. Build the individual pieces

We now implement the proposal pipeline as separate components:
1. research memo agent
2. outline agent
3. critique agent

These pieces correspond to the context-engineering techniques from the lecture:
- **write context**: agents save notes and drafts to `/workspace/...`
- **select context**: the research agent pulls only relevant web evidence
- **compress context**: the memo and outline are concise handoff artifacts
- **isolate context**: each component gets only the skill set it actually needs


In [55]:
class ResearchMemo(BaseModel):
    comparable_systems: list[str] = Field(default_factory=list)
    candidate_datasets_or_apis: list[str] = Field(default_factory=list)
    benchmarks_and_metrics: list[str] = Field(default_factory=list)
    feasibility_notes: list[str] = Field(default_factory=list)
    open_questions: list[str] = Field(default_factory=list)
    source_urls: list[str] = Field(default_factory=list)


class ProposalOutline(BaseModel):
    proposed_title: str
    problem: str
    data_plan: str
    planned_methods_and_models: str
    multi_agent_workflow: str
    evaluation_plan: str


class ReviewFeedback(BaseModel):
    missing_items: list[str] = Field(default_factory=list)
    weak_claims: list[str] = Field(default_factory=list)
    improvement_actions: list[str] = Field(default_factory=list)
    verdict: str


class ProposalDraft(BaseModel):
    proposed_title: str
    problem: str
    data_and_collection: str
    methods_and_models: str
    multi_agent_workflow: str
    evaluation_plan: str
    risks_and_open_questions: str
    source_urls: list[str] = Field(default_factory=list)
    final_markdown: str


def ensure_text_file(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content.strip() + "\n", encoding="utf-8")


def memo_to_markdown(memo: ResearchMemo) -> str:
    sections = {
        "Comparable systems": memo.comparable_systems,
        "Candidate datasets or APIs": memo.candidate_datasets_or_apis,
        "Benchmarks and metrics": memo.benchmarks_and_metrics,
        "Feasibility notes": memo.feasibility_notes,
        "Open questions": memo.open_questions,
        "Source URLs": memo.source_urls,
    }
    lines = ["# Research memo", ""]
    for heading, items in sections.items():
        lines.append(f"## {heading}")
        if items:
            lines.extend([f"- {item}" for item in items])
        else:
            lines.append("- None")
        lines.append("")
    return "\n".join(lines)


def outline_to_markdown(outline: ProposalOutline) -> str:
    sections = {
        "Problem": outline.problem,
        "Data and data collection": outline.data_plan,
        "Planned methods and models": outline.planned_methods_and_models,
        "Multi-agent workflow": outline.multi_agent_workflow,
        "Evaluation plan": outline.evaluation_plan,
    }
    lines = [f"# {outline.proposed_title}", ""]
    for heading, paragraph in sections.items():
        lines.append(f"## {heading}")
        lines.append(paragraph.strip())
        lines.append("")
    return "\n".join(lines)


def review_to_markdown(review: ReviewFeedback) -> str:
    sections = {
        "Missing items": review.missing_items,
        "Weak claims": review.weak_claims,
        "Improvement actions": review.improvement_actions,
    }
    lines = [f"# Review verdict: {review.verdict}", ""]
    for heading, items in sections.items():
        lines.append(f"## {heading}")
        if items:
            lines.extend([f"- {item}" for item in items])
        else:
            lines.append("- None")
        lines.append("")
    return "\n".join(lines)


In [56]:
def build_research_agent():
    return create_deep_agent(
        model=MODEL_NAME,
        tools=proposal_tools,
        backend=make_backend(),
        skills=[SHARED_SKILLS, RESEARCH_SKILLS],
        response_format=ResearchMemo,
        system_prompt=(
            "You are the research stage of an IE686 project-proposal pipeline. "
            "Always start by planning with write_todos for non-trivial tasks. "
            "Use live web research, capture exact benchmark and system names, and save a concise memo to "
            "/workspace/research/research_memo.md before returning. "
            "Look for recent scientific writing when it helps make the proposal more concrete."
        ),
        name="proposal_research_agent",
    )


def build_outline_agent():
    return create_deep_agent(
        model=MODEL_NAME,
        tools=[],
        backend=make_backend(),
        skills=[SHARED_SKILLS, MAIN_SKILLS],
        response_format=ProposalOutline,
        system_prompt=(
            "You convert research notes into a concrete project outline. "
            "Read /workspace/research/research_memo.md, apply the rubric skill, and save the draft to "
            "/workspace/outline/project_outline.md before returning. "
            "Write it as a compact executive summary in prose, roughly three pages in length. "
            "Include source information and a short references section based on the sources actually used."
        ),
        name="proposal_outline_agent",
    )


def build_critic_agent():
    return create_deep_agent(
        model=MODEL_NAME,
        tools=[],
        backend=make_backend(),
        skills=[SHARED_SKILLS, REVIEW_SKILLS],
        response_format=ReviewFeedback,
        system_prompt=(
            "You critique IE686 proposal drafts. "
            "Read /workspace/outline/project_outline.md, apply the rubric and critic skills, "
            "and save actionable feedback to /workspace/review/review_feedback.md before returning."
        ),
        name="proposal_critic_agent",
    )


In [57]:
reset_workspace()

research_agent = build_research_agent()
research_prompt = textwrap.dedent(
    f"""
    Prepare a research memo for this project topic:
    {DEFAULT_TOPIC}

    Find concrete comparable systems, datasets or APIs, benchmarks, and evaluation metrics.
    Look for recent scientific writing when it materially sharpens the proposal.
    Keep the memo focused on what would help a student team write a realistic project outline.
    """
).strip()

research_result, research_run_url = invoke_with_trace(
    research_agent,
    research_prompt,
    run_name="proposal-piece-research",
)

research_memo = research_result["structured_response"]
ensure_text_file(WORKSPACE_ROOT / "research" / "research_memo.md", memo_to_markdown(research_memo))

display(pd.Series(research_memo.model_dump(), name="value").to_frame())
print("Research run:", research_run_url or "LangSmith URL unavailable")


,value
comparable_systems,[Calendar.help (Microsoft Research) — workflow...
candidate_datasets_or_apis,[Enron Email Dataset (Enron corpus) — large pu...
benchmarks_and_metrics,"[Inbox triage: per-class precision/recall/F1, ..."
feasibility_notes,[Privacy & consent: live evaluation against re...
open_questions,[Scope of automation vs human review: which ag...
source_urls,[https://www.microsoft.com/en-us/research/proj...


Research run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d005c-37b3-75e1-9875-b58dd550724a?poll=true


In [58]:
outline_agent = build_outline_agent()
outline_prompt = textwrap.dedent(
    f"""
    Read /workspace/research/research_memo.md and draft the project outline for:
    {DEFAULT_TOPIC}

    Answer the four official outline questions.
    Write the result as a compact executive summary in prose, roughly three pages in length, not as a bullet list.
    Include source information and a short references section based on the sources actually used.
    """
).strip()

outline_result, outline_run_url = invoke_with_trace(
    outline_agent,
    outline_prompt,
    run_name="proposal-piece-outline",
)

outline = outline_result["structured_response"]
ensure_text_file(WORKSPACE_ROOT / "outline" / "project_outline.md", outline_to_markdown(outline))

display(Markdown(outline_to_markdown(outline)))
print("Outline run:", outline_run_url or "LangSmith URL unavailable")


# Secretary Agent with OpenClaw: Inbox Triage, Calendar Coordination, Meeting Preparation, and Follow-up Tracking

## Problem
Knowledge workers spend substantial time on routine coordination tasks—triaging email, negotiating meeting times across constraints, preparing for meetings, and tracking post-meeting action items—yet current automation addresses these needs only piecemeal. Production systems demonstrate parts of this pipeline (automated reply suggestions in Gmail, meeting transcription/summarization services, and hybrid scheduling services such as Calendar.help), but there is no open, config-first agent architecture shown to integrate end-to-end assistance across triage, scheduling, meeting brief generation, and follow-up tracking while preserving user trust and safe human-in-the-loop control. The project therefore seeks to design and implement an OpenClaw-based “secretary” agent that reduces time-on-task and manual work for realistic office workflows, minimizes harmful automated actions (e.g., mistaken auto-archives or calendar conflicts), and provides measurable improvements compared to manual baselines and hybrid scheduling systems.

## Data and data collection
Development and evaluation will combine public corpora, provider APIs for prototype integration, and a small consented modern data slice for final validation. Offline model training and component benchmarks will use the Enron email corpus (Stanford SNAP mirror) and the Avocado Research Email Collection (LDC2015T03) to train triage classifiers and date/time extraction; meeting summarization and action-item extraction experiments will use the AMI and ICSI meeting corpora. To evaluate end-to-end behavior with real integrations, we will prototype against Gmail and Google Calendar APIs (and/or Microsoft Graph for Outlook) after obtaining participant consent and IRB approvals as needed. Where robustness to ASR noise is required, we will evaluate on AMI/ICSI transcripts plus synthetically degraded transcripts, and we will record a small set of consenting meetings processed through common ASR services. All live data will be handled under strict privacy controls (minimal retention, encrypted storage, and explicit opt-in). If live consented data proves unavailable, the design includes an offline simulation mode using mock provider APIs; every dataset and API choice is traceable to evidence in the project research notes.

## Planned methods and models
The system will be a hybrid pipeline combining supervised classifiers, prompt-engineered and fine-tuned LLMs, and rule-based extractors. Inbox triage will be framed as a multi-class classification task (action now / later / archive / auto-respond / delegate) using transformer-based classifiers fine-tuned on labeled slices of Enron and Avocado plus a small modern annotation set; metrics will include per-class precision/recall and macro-F1. Short-response suggestion and assisted composition will follow the Smart Reply / Smart Compose interaction model: candidate replies generated by lightweight models or LLM prompts and ranked for user confirmation or edit. Scheduling will rely on intent and temporal-entity extraction trained or adapted from email-event-extraction datasets and a negotiation policy that minimizes the number of messages to confirm a meeting while respecting calendar constraints. Pre-meeting briefs and post-meeting summaries/action items will be produced by LLM summarization over calendar context, documents, and transcripts (evaluating both extractive and abstractive approaches), with robustness testing against ASR error. Follow-up detection will use sequence labeling and relation extraction to identify tasks, owners, and deadlines and record them in a task tracker; the default user-facing behavior will be suggest-and-confirm with opt-in auto-apply for low-risk actions. Model choices will be pragmatic: compact fine-tuned models or open weights for latency-sensitive triage, and larger API-based LLMs for higher-quality summarization where cost and latency permit. Human-in-the-loop fallbacks and microtasking patterns (inspired by Calendar.help) will be used for ambiguous or high-risk cases.

## Multi-agent workflow
Using OpenClaw’s skill-oriented, config-first architecture, the assistant will be decomposed into cooperating skills orchestrated by a coordination layer. Core skills include an Ingest skill (pulls emails and calendar events via provider APIs and normalizes context), a Triage skill (classifies messages and generates candidate actions and reply drafts), a Scheduler skill (extracts availability, proposes times, negotiates constraints, and writes events via Calendar APIs), a Meeting Briefing skill (compiles pre-meeting context, generates briefs, and creates post-meeting summaries and action-item extractions from transcripts), and a Task-tracker skill (creates and surfaces follow-up tasks and reminders). An orchestration agent routes inputs, applies business rules (privacy filters, rate limits, human-confirmation thresholds), and implements handoffs: suggestions are surfaced through OpenClaw channels (email, chat, or a web dashboard) where users can accept or edit; fallback microtasks or human review are invoked for ambiguous or risky operations. The system will support both an offline simulation mode (mocked provider APIs for reproducibility) and a live integration mode (consented accounts), enabling controlled comparisons and iterative rollback of automation thresholds during evaluation.

## Evaluation plan
Evaluation will combine component-level metrics, end-to-end workflow measures, and user-centered studies. Component metrics: triage per-class precision/recall/F1 and macro-F1; top-K suggestion accuracy for reply generation; scheduling metrics such as number of messages to confirm, time-to-scheduled, and scheduling success rate without human rework; and precision/recall for action-item extraction and owner/deadline detection. Offline evaluations will use Enron, Avocado, AMI, and ICSI corpora as appropriate. End-to-end evaluation will be a controlled within-subjects user study (recommended 6–12 participants) simulating a 1–2 hour realistic work session comprising an inbox triage block, multiple constrained scheduling tasks, a mock meeting with provided transcript, and follow-up task handling. Primary user metrics will include time-on-task, manual corrections, emails-to-confirm per scheduled meeting, task acceptance rate for agent-created tasks, SUS for usability, and NASA-TLX for cognitive load; qualitative interviews will probe trust and desired automation thresholds. Baselines include a manual workflow (standard Gmail/Calendar) and a hybrid human-assisted scheduling baseline inspired by Calendar.help. All live trials will require informed consent and adhere to privacy and ethical review requirements; open questions and assumptions (e.g., limits of Enron/Avocado as modern proxies, exact auto-apply policies) will be treated as experimental variables documented in results.


Outline run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d005e-f2b0-7783-9140-9e13bc7c4ae9?poll=true


In [59]:
critic_agent = build_critic_agent()
critic_prompt = (
    "Review /workspace/outline/project_outline.md against the course outline rubric. "
    "Identify missing specificity, unsupported claims, weak evaluation, or places where the draft falls back into a bullet-list style."
)

critic_result, critic_run_url = invoke_with_trace(
    critic_agent,
    critic_prompt,
    run_name="proposal-piece-critic",
)

review = critic_result["structured_response"]
ensure_text_file(WORKSPACE_ROOT / "review" / "review_feedback.md", review_to_markdown(review))

display(Markdown(review_to_markdown(review)))
print("Critic run:", critic_run_url or "LangSmith URL unavailable")


# Review verdict: Revision recommended. The draft covers required sections and shows coherent scope, but needs targeted specificity: quantitative success criteria, annotation/sample-size details, baseline implementation descriptions, privacy/IRB and contingency plans, cost/timeline estimates, and conversion of listy sections into executive-summary prose. Address these items and resubmit.

## Missing items
- Quantitative success criteria and statistical power/sample-size calculation
- Detailed annotation protocol: labels, guidelines, annotator counts, inter-annotator agreement targets
- Target dataset sizes and train/validation/test splits
- Exact baseline implementations (how Gmail/Calendar baseline and Calendar.help-style baseline will be realized)
- Concrete confidence thresholds and policy for auto-apply vs suggest-and-confirm
- Contingency/fallback plan if OpenClaw lacks required integration hooks
- Concrete IRB/consent process, data retention periods, and encryption details
- Compute, cost, and timeline estimates (training/inference and API usage costs)
- Reproducibility/release plan (code, configs, mock providers, checkpoints)
- ASR robustness test details (noise levels, WER targets, synthesis method)
- Recruitment plan and inclusion/exclusion criteria for user studies
- Statistical test plan and effect-size targets for each primary metric
- Failure modes and rollback procedures for erroneous automated actions

## Weak claims
- "meaningfully reduces time-on-task and manual work" — asserted without quantitative targets or preliminary evidence
- "OpenClaw provides the necessary integration hooks" — assumed without explicit verification or contingency
- "Datasets and APIs are chosen for traceability to prior work" — stated but not tied to explicit citations/versions for each claim
- "The default will be suggest-and-confirm; hybrid microtasking patterns from Calendar.help will inform a fallback" — vague on how human fallback will be implemented and measured
- "small fine-tuned transformers or open models for on-premise inference" — model families and selection criteria not specified

## Improvement actions
- Add explicit, testable primary and secondary hypotheses (e.g., % reduction in time-on-task) and perform a power/sample-size calculation to justify participant counts
- Specify annotation plan: number of examples per class, annotation guidelines, who will annotate, and target inter-annotator agreement
- Name candidate models (e.g., DistilBERT, RoBERTa, Llama2-7B), expected parameter sizes, and when API-based LLMs will be used vs on-prem models
- Describe exactly how baselines will be executed and measured (what features of Calendar.help are being compared, how Gmail baseline is configured)
- Define confidence scoring and thresholds for auto-apply vs suggest-and-confirm and add an experimental plan to vary thresholds
- Replace bullet-list sections (methods, skills, multi-agent workflow) with concise executive-summary prose integrating architecture and modes (offline simulation vs live)
- Provide explicit IRB/consent language, data retention windows, encryption-at-rest/transfer practices, and minimal retained artifacts list
- Add compute/cost estimates (GPU hours, expected API call costs) and a brief timeline/milestones table
- Specify ASR robustness experiments: how noisy transcripts are synthesized or recorded, WER targets, and evaluation metrics under noise
- Add a reproducibility statement listing what will be released (code, configs, mock provider), licensing, and reproduction instructions
- Include failure-mode handling: rollback, user notification, logging and audit trails for automated actions
- Add concrete statistical tests for each metric and define success thresholds (e.g., p-values, effect sizes)


Critic run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d0060-2bcc-7883-aa9a-05064f617e8c?poll=true


Try it out:
- Change the topic and rerun only the research cell.
- Inspect whether the outline and critique improve when the research memo becomes more specific.
- Open the run traces and compare the tool usage of the research agent versus the outline and critic agents.


## 8. Final deep agent with isolated subagents

The final capstone combines everything:
- live web tools,
- checked-in skills,
- file-backed notes,
- isolated subagents,
- final structured output,
- LangSmith trace.

The important design point is that **subagents do not automatically inherit every skill**.
We pass each subagent its own `skills=[...]` configuration.


In [60]:
def build_final_orchestrator():
    return create_deep_agent(
        model=MODEL_NAME,
        tools=proposal_tools,
        backend=make_backend(),
        skills=[SHARED_SKILLS, MAIN_SKILLS],
        response_format=ProposalDraft,
        system_prompt=(
            "You orchestrate an IE686 project-proposal writing pipeline. "
            "For any real proposal task you must first write todos, then delegate external research to the "
            "`researcher` subagent, delegate workflow and outline construction to `workflow_designer`, "
            "delegate draft review to `critic`, revise the draft, and save the final markdown to "
            "/workspace/final/final_proposal.md. "
            "Write the final draft as a compact executive summary in prose, roughly three pages in length. "
            "Only include datasets, benchmarks, systems, or recent scientific writing that are grounded in the gathered evidence. "
            "Include source information and a short references section based on the sources actually used."
        ),
        subagents=[
            {
                "name": "researcher",
                "description": "Researches comparable systems, datasets, APIs, benchmarks, and evaluation metrics for a proposed project topic.",
                "system_prompt": (
                    "You are the research specialist. Use the web tools, follow the benchmark-research skill, "
                    "look for recent scientific writing when it materially sharpens the proposal, "
                    "and write a concise memo to /workspace/research/research_memo.md. "
                    "Return a short summary with exact names and URLs."
                ),
                "tools": proposal_tools,
                "skills": [SHARED_SKILLS, RESEARCH_SKILLS],
            },
            {
                "name": "workflow_designer",
                "description": "Turns research notes into a concrete project outline with a plausible multi-agent workflow and evaluation plan.",
                "system_prompt": (
                    "You are the proposal drafting specialist. Read /workspace/research/research_memo.md, "
                    "apply the rubric and proposal-writer skills, and write the draft to "
                    "/workspace/outline/project_outline.md. Write the draft as a compact executive summary in prose, "
                    "not as a bullet list. Include source information and a short references section based on the "
                    "sources actually used. Return a concise summary of the structure."
                ),
                "tools": [],
                "skills": [SHARED_SKILLS, MAIN_SKILLS],
            },
            {
                "name": "critic",
                "description": "Critiques a proposal draft for missing rubric coverage, vague claims, and weak evaluation.",
                "system_prompt": (
                    "You are the proposal critic. Read /workspace/outline/project_outline.md or "
                    "/workspace/final/final_proposal.md, follow the proposal-critic skill, and write feedback to "
                    "/workspace/review/review_feedback.md. Return concise actionable feedback."
                ),
                "tools": [],
                "skills": [SHARED_SKILLS, REVIEW_SKILLS],
            },
        ],
        name="ie686_proposal_orchestrator",
    )


final_agent = build_final_orchestrator()


In [61]:
TOPIC = DEFAULT_TOPIC

final_prompt = textwrap.dedent(
    f"""
    Build a strong project proposal draft for this suggested topic:
    {TOPIC}

    Requirements:
    - answer the official four project-outline questions,
    - research comparable systems, candidate datasets or APIs, and suitable benchmarks or metrics,
    - use recent scientific writing when it helps make the proposal more concrete and current,
    - propose a concrete multi-agent workflow,
    - critique the draft and revise it once before finalizing,
    - include source information and a short references section based on the sources actually used,
    - write the final result as a compact executive summary in prose, roughly three pages in length,
    - save the final markdown draft to /workspace/final/final_proposal.md.
    """
).strip()

reset_workspace()
final_result, final_run_url = invoke_with_trace(
    final_agent,
    final_prompt,
    run_name="deepagents-final-proposal",
)

final_draft = final_result["structured_response"]
ensure_text_file(WORKSPACE_ROOT / "final" / "final_proposal.md", final_draft.final_markdown)

display(Markdown(final_draft.final_markdown))
print("Final LangSmith run:", final_run_url or "LangSmith URL unavailable")


# OpenClaw Secretary — Executive Summary

Modern knowledge work is dominated by time-consuming coordination: triaging an overflowing inbox, negotiating meeting times, preparing concise meeting briefs, and tracking follow-ups. These tasks are high-frequency, low-strategy, and often interruptive—making them ideal targets for partial automation. OpenClaw Secretary is a modular, privacy-conscious multi-agent assistant that automates core secretary functions (inbox triage, calendar coordination, meeting preparation, and follow-up tracking) while keeping humans in control. The system will be implemented on OpenClaw as the orchestration/runtime layer, integrating OAuth-backed connectors, a retrieval-backed knowledge store, and narrowly scoped specialist agents. Evaluation will combine reproducible automated benchmarks and controlled user studies reflecting realistic office workflows.

1) Problem (what we are solving)

Knowledge workers regularly spend substantial weekly time on coordination tasks: deciding which messages require action, negotiating meeting times across calendars, preparing distilled briefs before meetings, and ensuring assigned follow-ups are completed. The project reduces cognitive load and time spent on these routine tasks by building an assistant that: (a) triages incoming messages into actionable categories; (b) coordinates and confirms meetings with minimal negotiation rounds; (c) generates concise, source-cited meeting briefs and agendas; and (d) detects and tracks follow-up tasks to closure. The assistant emphasizes conservative defaults, auditable decision logs, and explicit human approval for side-effecting actions.

2) Data (what data, where, how gathered)

Planned data sources and access model:
- Public corpora for reproducible benchmarking: Enron Email Dataset (https://www.cs.cmu.edu/~enron/), Avocado Research Email collection (LDC2015T03), QMSum (https://github.com/Yale-LILY/QMSum), MeetingBank (https://meetingbank.github.io/), AMI/ICSI meeting corpora.
- Production integration: OAuth connectors to Gmail API, Google Calendar API, and Microsoft Graph for reading/acting on mail and calendar items. Transcription via AssemblyAI or Google Speech-to-Text for meeting audio.
- Pilot collection: opt-in, anonymized enterprise logs and consenting meeting recordings for fine-tuning and user-study realism (legal/IRB review required). All production data is subject to tenant policies and may use tenant-hosted inference and tenant-managed key management.

Data handling and labeling:
- Triage label schema: {action_required, meeting_request, informational, read_later, spam}. Target held-out test set ≈2,000 messages with ≥200 examples per non-spam class for benchmarking.
- Meeting brief annotations: target 300–500 annotated meeting briefs mapping transcript passages to concise, cited summaries.
- Follow-up annotation: 300–500 examples labeling action items, owners, and completion timestamps for span-level F1 evaluation.
- When internal data is unavailable, use augmented public/synthetic data to match label distributions and document the domain shift risk.

3) Solution and multi-agent workflow (methods, models, orchestration)

Architecture and agent roles (OpenClaw as orchestrator):
- Inbox Triage Agent: a classifier (fine-tuned DistilBERT/BERT) and entity/temporal extractor that labels messages and extracts candidate actions.
- Scheduler Agent: availability matcher and negotiation manager that proposes slots (first proposal + up to one counterproposal = negotiation round), interfaces with calendars via API connectors, and surfaces final confirmations to users for approval.
- Meeting Prep Agent: retrieval-augmented generator (RAG) that constructs concise pre-meeting briefs with explicit citations to thread passages, documents, and calendar metadata. Use Longformer/LED or T5-family models for long-context summarization when needed.
- Follow-up Tracker Agent: extracts action items from emails/transcripts, assigns owners, issues reminders, and tracks closure state.
- Integrator/Orchestrator Agent (OpenClaw): routes data, enforces policy and RBAC, records provenance and decision logs, manages connectors and caching, and exposes a minimal UI/API for human approvals.

Model strategy:
- Early iterations prioritize prompt engineering + RAG to reduce annotation cost and speed iteration. Fine-tuning is reserved for persistent failure modes: triage classification, sensitivity detection, and stylistic constraints for generated replies/briefs.
- Retrieval is backed by a vector store (FAISS/Pinecone/Weaviate) over embeddings; retrieval contexts are fed to the generator with provenance metadata for citations.
- Safety: PII detectors and NER-based redaction precede any external transmission; tenant-hosted inference and tenant KMS are supported for sensitive deployments.

Baselines and reproducibility:
- Baselines: rule-based triage, DistilBERT classifier, lead-3/PEGASUS/T5 summarizers, deterministic first-available scheduler, and a zero-shot LLM prompting baseline for generation.
- Public datasets (Enron, QMSum, MeetingBank, AMI) will serve reproducible baselines; results on pilot internal data will be reported separately with anonymization procedures described.

4) Evaluation plan (how success is measured)

Automated component metrics:
- Triage: per-class precision/recall/F1, confusion matrices, and calibration (Brier score). Initial target F1 ≥ 0.80 (to be validated in pilot).
- Summarization: ROUGE/L, BERTScore, QA-based factuality (QAGS/QAFactEval), and citation-precision (percent of claimed facts supported by retrieved passages). Human usefulness Likert ratings supplement automated scores.
- Scheduling: negotiation rounds to agreement, time-to-confirm, participant satisfaction (Likert), and % manual reschedule. Target: ≥75% of meeting requests scheduled within ≤2 negotiation rounds.
- Follow-up tracking: span-level F1 for action-item extraction, owner-assignment accuracy, and time-to-closure distribution.

User study (system-level):
- Design: counterbalanced within-subjects 2-week deployment; target N=20–30 knowledge workers; pilot N≈6 for feasibility and power estimation. Primary outcome: time saved per week (instrumented). Secondary: task completion latency, SUS, override frequency, and qualitative interviews.
- Hypotheses: measurable time savings (target ≥30 minutes/week to be validated), reduced task latencies, and SUS improvement relative to baseline.
- Ethics: IRB submission, informed consent, and tenant/legal sign-off for any internal log usage.

Success criteria (example thresholds to validate in pilot):
- Triage F1 ≥ 0.80; scheduling success ≥75% within ≤2 rounds; meeting briefs rated "useful" by ≥70% of users; measurable average time saving per user (pilot-determined).

Timeline, resources, and risks

- Timeline (~6 months): months 0–1 connectors, data & policy framework; months 1–3 agent development and RAG prompt iterations; months 3–4 fine-tuning and synthetic benchmark creation; months 4–5 internal pilot; month 6 user study and analysis.
- Team: 4-person core (ML engineer, backend/infra, UX/designer, research assistant/annotator). Compute: intermittent 1–2 GPU-class instances for fine-tuning; production depends on tenant hosting choices.
- Annotation: ~1,000–3,000 triage labels initially, 300–500 meeting brief and follow-up annotations.
- Primary risks and mitigations: privacy/legal constraints (mitigate via opt-in, tenant-hosted inference, KMS, and IRB); hallucination (RAG, citation-precision checks, human approvals); adoption/trust (conservative defaults, editable drafts, provenance UI).

Sources and short references (selected)

- Toolformer — https://arxiv.org/abs/2302.04761 (self-supervised tool use)
- ReAct — https://arxiv.org/abs/2210.03629 (reasoning + acting pattern)
- Reflexion — https://arxiv.org/abs/2303.11366 (iterative agent improvement)
- RAG — https://arxiv.org/abs/2005.11401 (retrieval-augmented generation)
- QMSum — https://github.com/Yale-LILY/QMSum and https://aclanthology.org/2021.naacl-main.472.pdf (meeting summarization)
- MeetingBank — https://meetingbank.github.io/ and https://aclanthology.org/2023.acl-long.906.pdf (large meeting dataset)
- Enron Email Dataset — https://www.cs.cmu.edu/~enron/ (public email corpus)
- Avocado Research Email Collection (LDC2015T03) — https://catalog.ldc.upenn.edu/LDC2015T03
- AMI Meeting Corpus — https://www.openslr.org/16/
- Gmail API — https://developers.google.com/workspace/gmail/api
- Microsoft Graph — https://learn.microsoft.com/en-us/graph/overview
- QAGS (factuality metric) — https://aclanthology.org/2020.acl-main.450.pdf

Deliverables

- A reproducible prototype on OpenClaw with modular agents for triage, scheduling, meeting briefs, and follow-ups; connectors for Gmail/Calendar/Microsoft Graph; RAG-backed generation with citation provenance; and a human approval UI.
- A benchmark suite combining public datasets (Enron, QMSum, MeetingBank, AMI) and an anonymized internal pilot dataset (where permitted), with code and evaluation scripts for triage, summarization, scheduling, and follow-up extraction.
- A controlled within-subjects user study report with quantitative and qualitative analysis.

Next steps

- Finalize IRB/legal approvals for pilot data collection and tenant-hosting options.
- Implement connectors and a minimal triage + scheduler prototype on OpenClaw for a 6-week pilot.
- Run the pilot, collect annotated examples, and iterate on prompts and fine-tuning for persistent failure modes.

This proposal provides a concrete, evidence-grounded path to building and evaluating OpenClaw Secretary while prioritizing privacy, auditable decision-making, and realistic office workflows.


Final LangSmith run: https://smith.langchain.com/o/6ce8d416-0467-420e-bc34-dfa94d96e7b1/projects/p/c173cd8c-7d37-4fc8-92e1-d52192193da0/r/019d0060-fd9f-7bd3-b6e6-aff77b5fca2c?poll=true


In [62]:
artifact_rows = []
for path in sorted(WORKSPACE_ROOT.rglob("*")):
    if path.name == ".gitkeep":
        continue
    artifact_rows.append(
        {
            "path": str(path.relative_to(ROOT)),
            "is_dir": path.is_dir(),
            "bytes": path.stat().st_size if path.is_file() else 0,
        }
    )

pd.DataFrame(artifact_rows)


,path,is_dir,bytes
0,deepagents_lab/workspace/final,True,0
1,deepagents_lab/workspace/final/final_proposal.md,False,9875
2,deepagents_lab/workspace/outline,True,0
3,deepagents_lab/workspace/outline/project_outli...,False,13978
4,deepagents_lab/workspace/research,True,0
5,deepagents_lab/workspace/research/research_mem...,False,10748
6,deepagents_lab/workspace/review,True,0
7,deepagents_lab/workspace/review/review_feedbac...,False,12922
8,deepagents_lab/workspace/scratch,True,0


In [63]:
for relative in [
    "research/research_memo.md",
    "outline/project_outline.md",
    "review/review_feedback.md",
    "final/final_proposal.md",
]:
    path = WORKSPACE_ROOT / relative
    if path.exists():
        print(f"\n=== {relative} ===\n")
        print(path.read_text(encoding="utf-8")[:4000])



=== research/research_memo.md ===

{
  "title": "Research memo: Building a 'Secretary Agent' with OpenClaw",
  "date": "2026-03-18",
  "1_comparable_systems": [
    {
      "name": "Superhuman",
      "capabilities": "Email triage, fast inbox navigation, follow-up reminders, snippets; limited scheduling integrations",
      "stage": "production",
      "url": "https://superhuman.com"
    },
    {
      "name": "SaneBox",
      "capabilities": "Automated inbox triage (folders: SaneLater), snooze, follow-up reminders",
      "stage": "production",
      "url": "https://www.sanebox.com"
    },
    {
      "name": "Clara Labs (Clara)",
      "capabilities": "Human+AI scheduling assistant that manages meeting coordination and reminders",
      "stage": "production/service",
      "url": "https://claralabs.com/ (historical service; company site)")
    },
    {
      "name": "Calendly",
      "capabilities": "Automated scheduling and calendar coordination (focus: booking), integrates with ca

What to inspect in LangSmith for the final run:
- Did the main agent call `task` and delegate to the correct subagent?
- Which skills appear to have guided the research and critique phases?
- Which files were written into `/workspace/`?
- Where did the revision loop improve the draft?


## 9. Final try-it-out

Replace `TOPIC` with your own assigned student-project topic and rerun the final orchestration cells.

Good variants:
- `Text-to-SQL`
- `Online Shopping Assistant`
- `Text-to-BPMN`
- a more specific subtopic inside one of those areas

When you change the topic, keep the skills fixed first. Only after that should you edit the skills and inspect how the traces change.
